# 03 · Feature engineering: demand, service and geography

**Question:** Which features describe published demand, scheduled service and route structure?

This notebook builds interpretable tables for the Manhattan feed, preserving exact route identities,
calendar selection and source versions. Demand-only history remains available for all collected routes.
Hourly demand uses **published route-hour boarding estimates**. Stop-level and
direction-level passenger allocations are unavailable.

Run after notebook 02. This is preparation through EDA, not fitted preprocessing or model training.

In [1]:
from pathlib import Path
import sys, json, hashlib, zipfile
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts/collect_new_york.py').exists())
sys.path.insert(0, str(ROOT / 'scripts'))
RAW = ROOT / 'data/raw/new_york'
OUT = ROOT / 'data/processed/new_york'
REPORT = ROOT / 'reports/eda'
OUT.mkdir(parents=True, exist_ok=True)
REPORT.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 90)
print('Project located; analysis uses local, frozen source files.')

Project located; analysis uses local, frozen source files.


## 1. Verify inputs and inspect the working grains

Daily and hourly tables have different observed windows. Historical schedules come from five
captures; six current borough feeds remain inspection/context data and are not applied retrospectively.

In [2]:
manifest = json.loads((OUT / 'cleaning_manifest.json').read_text())
for name, item in manifest['products'].items():
    assert hashlib.sha256((OUT / f'{name}.csv').read_bytes()).hexdigest() == item['sha256']
daily = pd.read_csv(OUT / 'daily_clean.csv', dtype={'bus_route':'string'}, parse_dates=['payment_date'])
hourly = pd.read_csv(OUT / 'hourly_clean.csv', dtype={'bus_route':'string'}, parse_dates=['payment_date', 'local_hour'])
crosswalk = pd.read_csv(OUT / 'route_crosswalk.csv', dtype='string')
pilot_routes = sorted(crosswalk.bus_route.unique())
display(pd.DataFrame([['daily', len(daily), daily.payment_date.min(), daily.payment_date.max()],
                      ['hourly', len(hourly), hourly.payment_date.min(), hourly.payment_date.max()]],
                     columns=['table', 'rows', 'first_date', 'last_date']))
print('Historically referenced Manhattan routes:', len(pilot_routes))

,table,rows,first_date,last_date
0,daily,275896,2025-01-01,2026-09-30
1,hourly,993600,2026-07-01,2026-09-30


Historically referenced Manhattan routes: 41


## 2. Calendar features and exact-date historical lags

Weekday/Saturday/Sunday describe the calendar, not an inferred agency holiday category. Calendar
exceptions are handled separately when selecting scheduled trips. Lags use the **same route at
exactly t−1, t−7, t−14, t−21 and t−28 days**; missing dates never become previous-row substitutions.

These are retrospective candidate features. Publication delay is unknown, so a t−1 count is not
assumed available the next morning. No split, fitted scaler or model is created in this stage.

In [3]:
from analysis_utils import exact_date_lags, haversine_km, trip_clock_hours
from validate_historical_alignment import read_feed, expand_date, block_conflicts
daily['day_of_week'] = daily.payment_date.dt.dayofweek
daily['month'] = daily.payment_date.dt.month
daily['is_weekend'] = daily.day_of_week.ge(5)
daily['day_type'] = np.select([daily.day_of_week.eq(5), daily.day_of_week.eq(6)], ['Saturday','Sunday'], default='Weekday')
daily_features = exact_date_lags(daily)
hourly['hour'] = hourly.local_hour.dt.hour
hourly['day_of_week'] = hourly.payment_date.dt.dayofweek
hourly['is_weekend'] = hourly.day_of_week.ge(5)
hourly['day_type'] = np.select([hourly.day_of_week.eq(5), hourly.day_of_week.eq(6)], ['Saturday','Sunday'], default='Weekday')
display(daily_features.loc[daily_features.bus_route.eq('M1'),
    ['bus_route','payment_date','published_boardings','analysis_boardings','day_type','lag_1d','lag_7d','past_same_weekday_mean']].head(12))
display(daily_features.filter(regex='^lag_|seasonal|past_same').isna().sum().rename('unavailable_values').to_frame())

,bus_route,payment_date,published_boardings,analysis_boardings,day_type,lag_1d,lag_7d,past_same_weekday_mean
145,M1,2025-01-01,3007,3007.0,Weekday,NaN,NaN,NaN
570,M1,2025-01-02,7242,7242.0,Weekday,3007.0,NaN,NaN
995,M1,2025-01-03,7029,7029.0,Weekday,7242.0,NaN,NaN
1420,M1,2025-01-04,5728,5728.0,Saturday,7029.0,NaN,NaN
1845,M1,2025-01-05,3817,3817.0,Sunday,5728.0,NaN,NaN
2270,M1,2025-01-06,7810,7810.0,Weekday,3817.0,NaN,NaN
2695,M1,2025-01-07,8483,8483.0,Weekday,7810.0,NaN,NaN
3120,M1,2025-01-08,8534,8534.0,Weekday,8483.0,3007.0,NaN
3545,M1,2025-01-09,8496,8496.0,Weekday,8534.0,7242.0,NaN
3970,M1,2025-01-10,7785,7785.0,Weekday,8496.0,7029.0,NaN


,unavailable_values
lag_1d,3462
lag_7d,6162
lag_14d,9312
lag_21d,12462
lag_28d,15612
seasonal_history_count,0
past_same_weekday_mean,6300


## 3. Trip durations, stop patterns and shape lengths for each archive

Order stop visits by numeric sequence. Duration is final arrival minus first departure, preserving
times beyond 24:00. A trip's shape is joined by ID within its own archive. Length sums haversine
segments along the supplied polyline; it is an approximation, not an independently verified road distance.

A stop-pattern hash preserves ordered repeated stops. Pattern-aware departure gaps later avoid
mixing opposite directions, different origins or different stopping patterns.

In [4]:
feeds, trip_products, shape_products, archive_audit = [], [], [], []
selected_geometry = {}
for path in sorted((RAW / 'gtfs/archive').glob('*.zip')):
    feed = read_feed(path)  # also verifies receipt, syntax, keys, references and calendars
    with zipfile.ZipFile(path) as z:
        st = pd.read_csv(z.open('stop_times.txt'), dtype={'trip_id':'string','stop_id':'string'})
        sh = pd.read_csv(z.open('shapes.txt'), dtype={'shape_id':'string'})
        stops = pd.read_csv(z.open('stops.txt'), dtype={'stop_id':'string'})
    st = st.sort_values(['trip_id','stop_sequence'])
    endpoints = st.groupby('trip_id').agg(origin_stop_id=('stop_id','first'), final_stop_id=('stop_id','last'),
        stop_visits=('stop_id','size'), distinct_stops=('stop_id','nunique'))
    patterns = st.groupby('trip_id').stop_id.agg(tuple)
    endpoints['pattern_id'] = patterns.map(lambda seq: hashlib.sha256(json.dumps(seq).encode()).hexdigest()[:20])
    sh = sh.sort_values(['shape_id','shape_pt_sequence'])
    assert not sh.duplicated(['shape_id','shape_pt_sequence']).any()
    assert sh.shape_pt_lat.between(-90,90).all() and sh.shape_pt_lon.between(-180,180).all()
    prev = sh.groupby('shape_id')[['shape_pt_lat','shape_pt_lon']].shift()
    sh['segment_km'] = haversine_km(prev.shape_pt_lat, prev.shape_pt_lon, sh.shape_pt_lat, sh.shape_pt_lon)
    lengths = sh.groupby('shape_id').agg(shape_length_km=('segment_km',lambda s: s.sum(min_count=1)),
        largest_segment_km=('segment_km','max'), shape_points=('shape_id','size'))
    lengths['shape_review_required'] = lengths.shape_points.lt(2) | lengths.shape_length_km.le(0) | lengths.largest_segment_km.gt(5)
    trips = feed['trips'].merge(endpoints, on='trip_id', validate='one_to_one').merge(
        lengths, on='shape_id', how='left', validate='many_to_one')
    trips['bus_route'] = trips.route_id
    trips['scheduled_duration_minutes'] = (trips.final_arrival_seconds-trips.first_departure_seconds)/60
    trips['shape_review_required'] = trips.shape_review_required.fillna(True)
    trips['analysis_shape_length_km'] = trips.shape_length_km.mask(trips.shape_review_required)
    assert trips.duration_valid.all(), 'Invalid durations require incomplete-hours policy before continuation'
    assert trips.bus_route.isin(pilot_routes).all()
    trips['snapshot_id'] = feed['id']
    feed['trips'] = trips
    feeds.append(feed)
    trip_products.append(trips)
    shape_products.append(lengths.reset_index().assign(snapshot_id=feed['id']))
    archive_audit.append({'snapshot_id':feed['id'], 'capture_date':feed['capture'].date(),
        'trip_templates':len(trips), 'routes':trips.bus_route.nunique(), 'shapes':len(lengths),
        'shape_review_count':int(lengths.shape_review_required.sum()), 'max_time_seconds':feed['max_time_seconds']})
    if path.stem == 'manhattan_20260903':
        selected_geometry = {'stops':stops, 'stop_times':st[['trip_id','stop_id','stop_sequence']], 'shapes':sh}
    print('Prepared', path.stem)
trip_features = pd.concat(trip_products, ignore_index=True)
shape_features = pd.concat(shape_products, ignore_index=True)
display(pd.DataFrame(archive_audit))
display(trip_features[['snapshot_id','bus_route','trip_id','scheduled_duration_minutes','stop_visits','shape_length_km']].head())

Prepared manhattan_20251230


Prepared manhattan_20260327


Prepared manhattan_20260409


Prepared manhattan_20260624


Prepared manhattan_20260903


,snapshot_id,capture_date,trip_templates,routes,shapes,shape_review_count,max_time_seconds
0,manhattan_20251230,2025-12-30,37864,41,160,0,96180
1,manhattan_20260327,2026-03-27,37864,41,160,0,96180
2,manhattan_20260409,2026-04-09,30289,41,155,0,96180
3,manhattan_20260624,2026-06-24,30285,41,151,0,96180
4,manhattan_20260903,2026-09-03,37872,41,158,0,96180


,snapshot_id,bus_route,trip_id,scheduled_duration_minutes,stop_visits,shape_length_km
0,manhattan_20251230,M60+,MQ_H6-Weekday-009000_SBS60_901,40.0,19,14.761917
1,manhattan_20251230,M60+,MQ_H6-Weekday-013000_SBS60_901,41.0,17,15.553344
2,manhattan_20251230,M60+,MQ_H6-Weekday-019000_SBS60_901,38.0,19,14.761917
3,manhattan_20251230,M60+,MQ_H6-Weekday-022800_SBS60_901,39.0,17,15.553344
4,manhattan_20251230,M60+,MQ_H6-Weekday-029500_SBS60_906,41.0,19,14.761917


## 4. Expand the latest whole archive before calculating supply

Select the latest capture date on or before the service date. Apply feed validity, weekly flags and
exceptions. Never fall back to an older route/feed if the selected archive is future-effective.
Capture date is a catalog observation, not proven intraday publication availability.

Count trips only after calendar expansion. Preserve source **service date** in the pattern table.
Compute gaps within route + direction + origin + complete stop pattern. An undefined gap from a
single departure remains missing. Zero gaps are retained for simultaneous departures.

**Completeness check:** an active calendar does not necessarily cover the whole network. The output flags days with fewer than 20% of the same-archive/same-weekday median trip count (at least two comparison dates). Three boundary dates contain only two trips. Those dates and dependent incoming overnight comparisons are withheld, without substituting older feeds. This is a retrospective review screen, not a learned prediction feature.

In [5]:
first_date, last_date = pd.Timestamp('2026-01-01'), daily.payment_date.max()
calendar_rows, pattern_parts, clock_parts = [], [], []
for date in pd.date_range(first_date-pd.Timedelta(days=1), last_date+pd.Timedelta(days=1)):
    audit, active = expand_date(feeds, date)
    calendar_rows.append(audit)
    if active.empty:
        continue
    active = active.copy()
    active['block_overlap_review'] = active.route_short_name.isin(block_conflicts(active))
    keys = ['bus_route','direction_id','origin_stop_id','pattern_id']
    ordered = active.sort_values(keys+['first_departure_seconds'])
    ordered['headway_minutes'] = ordered.groupby(keys).first_departure_seconds.diff()/60
    profile = ordered.groupby(keys).agg(scheduled_departures=('trip_id','size'),
        first_departure_seconds=('first_departure_seconds','min'), last_departure_seconds=('first_departure_seconds','max'),
        mean_headway_minutes=('headway_minutes','mean'), median_headway_minutes=('headway_minutes','median'),
        min_headway_minutes=('headway_minutes','min'), max_headway_minutes=('headway_minutes','max'),
        headway_std_minutes=('headway_minutes','std'), headway_gap_count=('headway_minutes','count'),
        mean_duration_minutes=('scheduled_duration_minutes','mean'), median_duration_minutes=('scheduled_duration_minutes','median'),
        mean_stop_visits=('stop_visits','mean'), mean_shape_length_km=('analysis_shape_length_km','mean'),
        shape_review_required=('shape_review_required','any')).reset_index()
    profile['departure_span_hours'] = (profile.last_departure_seconds-profile.first_departure_seconds)/3600
    profile['service_date'] = date
    profile['source_snapshot'] = audit['source_snapshot']
    assert profile.scheduled_departures.sum() == len(active)
    assert profile.headway_gap_count.sum() == len(active)-len(profile)
    pattern_parts.append(profile)
    hours = trip_clock_hours(active, date)
    bad_routes = set(active.loc[active.block_overlap_review,'bus_route'])
    hours['block_overlap_review'] = hours.bus_route.isin(bad_routes)
    hours['donor_service_date'] = date
    hours['source_snapshot'] = audit['source_snapshot']
    clock_parts.append(hours)
calendar_audit = pd.DataFrame(calendar_rows)
calendar_audit['date'] = pd.to_datetime(calendar_audit.date)
calendar_audit['day_of_week'] = calendar_audit.date.dt.dayofweek
active_days = calendar_audit.loc[calendar_audit.status.eq('active')]
typical_counts = active_days.groupby(['source_snapshot','day_of_week']).scheduled_trips.agg(
    typical_same_snapshot_weekday_trips='median', comparable_service_dates='size').reset_index()
calendar_audit = calendar_audit.merge(typical_counts,on=['source_snapshot','day_of_week'],how='left',validate='many_to_one')
calendar_audit['partial_calendar_review'] = (calendar_audit.status.eq('active') &
    calendar_audit.comparable_service_dates.ge(2) &
    calendar_audit.scheduled_trips.lt(0.20*calendar_audit.typical_same_snapshot_weekday_trips))
assert max(feed['max_time_seconds'] for feed in feeds) < 2*86400, 'Extend donor-date context for trips beyond 48 hours'
patterns_daily = pd.concat(pattern_parts, ignore_index=True)
patterns_daily = patterns_daily.merge(calendar_audit[['date','partial_calendar_review']],
    left_on='service_date',right_on='date',validate='many_to_one').drop(columns='date')
clock_contributions = pd.concat(clock_parts, ignore_index=True)
display(calendar_audit.groupby('status').agg(dates=('date','size'), trip_templates=('scheduled_trips','sum')))
display(calendar_audit.loc[calendar_audit.partial_calendar_review])
display(patterns_daily.loc[patterns_daily.service_date.eq('2026-09-08')].head(8))
print('Every active day passed trip-count, headway-count and integrated-duration conservation checks.')

,dates,trip_templates
status,,
active,264,2078829
selected_snapshot_no_active_trips,5,0
selected_snapshot_outside_feed_info,6,0


,date,source_snapshot,status,active_services,scheduled_trips,day_of_week,typical_same_snapshot_weekday_trips,comparable_service_dates,partial_calendar_review
3,2026-01-03,manhattan_20251230,active,1,2,5,6921.0,12,True
101,2026-04-11,manhattan_20260409,active,1,2,5,6921.0,11,True
178,2026-06-27,manhattan_20260624,active,1,2,5,6921.0,10,True


,bus_route,direction_id,origin_stop_id,pattern_id,scheduled_departures,first_departure_seconds,last_departure_seconds,mean_headway_minutes,median_headway_minutes,min_headway_minutes,max_headway_minutes,headway_std_minutes,headway_gap_count,mean_duration_minutes,median_duration_minutes,mean_stop_visits,mean_shape_length_km,shape_review_required,departure_span_hours,service_date,source_snapshot,partial_calendar_review
32797,M1,0,400080,3f00da39baf466892bf1,50,18420,73980,18.897959,17.0,7.0,98.0,12.523719,49,86.540000,88.0,66.0,13.666963,False,15.433333,2026-09-08,manhattan_20260903,False
32798,M1,0,400080,c3f73dbfdf782409502c,11,59460,65580,10.200000,10.0,8.0,12.0,1.398412,10,79.909091,79.0,42.0,13.666961,False,1.700000,2026-09-08,manhattan_20260903,False
32799,M1,0,405527,6f6435adf783bd7c0873,44,30360,90300,23.232558,20.0,9.0,148.0,20.943049,43,78.636364,83.0,61.0,12.313026,False,16.650000,2026-09-08,manhattan_20260903,False
32800,M1,1,405284,4ecd8ef633ec9aac776d,10,24120,29820,10.555556,10.0,8.0,16.0,2.920236,9,76.000000,78.0,42.0,13.699896,False,1.583333,2026-09-08,manhattan_20260903,False
32801,M1,1,405284,ac5f9b6d847c6f39c005,51,14400,68400,18.000000,15.0,6.0,105.0,13.681508,50,89.764706,99.0,67.0,13.699898,False,15.000000,2026-09-08,manhattan_20260903,False
32802,M1,1,405284,c5ba5778ba7ec013800e,44,24840,86700,23.976744,20.0,10.0,169.0,23.785238,43,81.931818,90.0,61.0,12.216001,False,17.183333,2026-09-08,manhattan_20260903,False
32803,M10,0,401269,5b483247237d18d743e4,1,40080,40080,NaN,NaN,NaN,NaN,NaN,0,23.000000,23.0,19.0,3.362856,False,0.000000,2026-09-08,manhattan_20260903,False
32804,M10,0,403846,6d27ebe3e974106934f9,78,20220,94500,16.077922,15.0,10.0,35.0,5.855196,77,50.935897,52.0,43.0,8.265338,False,20.633333,2026-09-08,manhattan_20260903,False


Every active day passed trip-count, headway-count and integrated-duration conservation checks.


## 5. Align supply to payment clock dates, including overnight service

A 25:10 departure belongs to its original service day but starts at 01:10 on the following
ordinary clock date. Trip hours are split by overlap with clock-hour intervals, conserving duration.
Use the [GTFS noon-minus-12-hours convention](https://gtfs.org/documentation/schedule/reference/#field-types)
in `America/New_York`, converted through UTC.

Supply is unavailable when current or preceding service context is missing. DST transition dates
and neighbours are withheld because the ridership publisher's repeated-hour semantics are not
established. Overlapping vehicle blocks also require review. Diagnostic partial sums are retained;
they never become validated denominators. Full missing schedules are not interpreted as zero service.

In [6]:
hours_index = pd.date_range(first_date, last_date + pd.Timedelta(hours=23), freq='h')
grid = pd.MultiIndex.from_product([pilot_routes,hours_index], names=['bus_route','local_hour']).to_frame(index=False)
supply = clock_contributions.groupby(['bus_route','local_hour']).agg(
    partial_scheduled_departures=('scheduled_departures','sum'),
    partial_scheduled_service_hours=('scheduled_service_hours','sum'),
    block_overlap_review=('block_overlap_review','any')).reset_index()
grid = grid.merge(supply, on=['bus_route','local_hour'], how='left', validate='one_to_one')
grid['payment_date'] = grid.local_hour.dt.normalize()
date_audit = calendar_audit.set_index('date')
grid['current_calendar_status'] = grid.payment_date.map(date_audit.status)
grid['prior_calendar_status'] = (grid.payment_date-pd.Timedelta(days=1)).map(date_audit.status)
grid['source_snapshot'] = grid.payment_date.map(date_audit.source_snapshot)
grid['prior_source_snapshot'] = (grid.payment_date-pd.Timedelta(days=1)).map(date_audit.source_snapshot)
dates = pd.date_range(first_date-pd.Timedelta(days=2),last_date+pd.Timedelta(days=2))
offset = pd.Series([d.tz_localize('America/New_York').utcoffset().total_seconds() for d in dates], index=dates)
transition_dates = offset.index[offset.ne(offset.shift())][1:]
dst_review_dates = {t + pd.Timedelta(days=k) for t in transition_dates for k in [-2,-1,0]}
grid['dst_context_review'] = grid.payment_date.isin(dst_review_dates)
# A block conflict anywhere on this route/clock date makes the daily denominator review-only.
grid['block_overlap_review'] = grid.block_overlap_review.astype('boolean').fillna(False).astype(bool)
grid['block_overlap_review'] = grid.groupby(['bus_route','payment_date']).block_overlap_review.transform('any')
context_ok = grid.current_calendar_status.eq('active') & grid.prior_calendar_status.eq('active')
grid['partial_calendar_review'] = (grid.payment_date.map(date_audit.partial_calendar_review).astype('boolean').fillna(True)
    | (grid.payment_date-pd.Timedelta(days=1)).map(date_audit.partial_calendar_review).astype('boolean').fillna(True))
grid['schedule_comparison_eligible'] = context_ok & ~grid.partial_calendar_review & ~grid.dst_context_review & ~grid.block_overlap_review
for column in ['scheduled_departures','scheduled_service_hours']:
    # Zero means no contribution ONLY when both selected calendars are complete.
    grid[column] = grid['partial_'+column].fillna(0).where(grid.schedule_comparison_eligible)
schedule_daily = grid.groupby(['bus_route','payment_date'], as_index=False).agg(
    scheduled_departures=('scheduled_departures',lambda s:s.sum(min_count=1)),
    scheduled_service_hours=('scheduled_service_hours',lambda s:s.sum(min_count=1)),
    schedule_comparison_eligible=('schedule_comparison_eligible','all'),
    dst_context_review=('dst_context_review','any'), block_overlap_review=('block_overlap_review','any'),
    partial_calendar_review=('partial_calendar_review','any'),
    source_snapshot=('source_snapshot','first'), prior_source_snapshot=('prior_source_snapshot','first'))
assert schedule_daily.loc[~schedule_daily.schedule_comparison_eligible,'scheduled_departures'].isna().all()
display(schedule_daily.groupby('schedule_comparison_eligible').agg(route_dates=('bus_route','size'),dates=('payment_date','nunique')))
display(schedule_daily.loc[schedule_daily.dst_context_review | schedule_daily.block_overlap_review].head(10))

,route_dates,dates
schedule_comparison_eligible,,
False,823,23
True,10370,253


,bus_route,payment_date,scheduled_departures,scheduled_service_hours,schedule_comparison_eligible,dst_context_review,block_overlap_review,partial_calendar_review,source_snapshot,prior_source_snapshot
65,M1,2026-03-07,NaN,NaN,False,True,False,False,manhattan_20251230,manhattan_20251230
66,M1,2026-03-08,NaN,NaN,False,True,False,False,manhattan_20251230,manhattan_20251230
67,M1,2026-03-09,NaN,NaN,False,True,False,False,manhattan_20251230,manhattan_20251230
338,M10,2026-03-07,NaN,NaN,False,True,False,False,manhattan_20251230,manhattan_20251230
339,M10,2026-03-08,NaN,NaN,False,True,False,False,manhattan_20251230,manhattan_20251230
340,M10,2026-03-09,NaN,NaN,False,True,False,False,manhattan_20251230,manhattan_20251230
611,M101,2026-03-07,NaN,NaN,False,True,False,False,manhattan_20251230,manhattan_20251230
612,M101,2026-03-08,NaN,NaN,False,True,False,False,manhattan_20251230,manhattan_20251230
613,M101,2026-03-09,NaN,NaN,False,True,False,False,manhattan_20251230,manhattan_20251230
884,M102,2026-03-07,NaN,NaN,False,True,False,False,manhattan_20251230,manhattan_20251230


## 6. Join one route and clock date/hour at a time

Both directions' service is combined because demand is not observed by direction. Compare only
the same route and local date/hour. Boarding totals divided by scheduled trips/hours are descriptive
service-use indicators; they do not measure load factor, actual operated productivity or delays.
An hour can have ongoing trips but no new departures, so its per-departure ratio is undefined.

In [7]:
matched_daily = daily_features.loc[daily_features.bus_route.isin(pilot_routes)].merge(
    schedule_daily, on=['bus_route','payment_date'], how='left', validate='one_to_one')
matched_daily['comparison_eligible'] = matched_daily.schedule_comparison_eligible.astype('boolean').fillna(False) & matched_daily.analysis_boardings.notna()
matched_daily['boardings_per_scheduled_departure'] = (matched_daily.analysis_boardings /
    matched_daily.scheduled_departures.where(matched_daily.scheduled_departures.gt(0))).where(matched_daily.comparison_eligible)
matched_daily['boardings_per_scheduled_service_hour'] = (matched_daily.analysis_boardings /
    matched_daily.scheduled_service_hours.where(matched_daily.scheduled_service_hours.gt(0))).where(matched_daily.comparison_eligible)
matched_hourly = hourly.loc[hourly.bus_route.isin(pilot_routes)].merge(
    grid.drop(columns='payment_date'), on=['bus_route','local_hour'], how='left', validate='one_to_one')
matched_hourly['comparison_eligible'] = matched_hourly.schedule_comparison_eligible.astype('boolean').fillna(False) & matched_hourly.analysis_boardings.notna()
matched_hourly['boardings_per_scheduled_service_hour'] = (matched_hourly.analysis_boardings /
    matched_hourly.scheduled_service_hours.where(matched_hourly.scheduled_service_hours.gt(0))).where(matched_hourly.comparison_eligible)
display(pd.DataFrame([
    ['route/date',len(matched_daily),int(matched_daily.comparison_eligible.sum()),matched_daily.bus_route.nunique()],
    ['route/hour',len(matched_hourly),int(matched_hourly.comparison_eligible.sum()),matched_hourly.bus_route.nunique()]
],columns=['grain','observed_rows','eligible_rows','route_codes']))
display(matched_daily.loc[matched_daily.comparison_eligible,
    ['bus_route','payment_date','published_boardings','scheduled_departures','scheduled_service_hours',
     'boardings_per_scheduled_departure','source_snapshot']].head(10))

,grain,observed_rows,eligible_rows,route_codes
0,route/date,26158,10083,41
1,route/hour,90528,86486,41


,bus_route,payment_date,published_boardings,scheduled_departures,scheduled_service_hours,boardings_per_scheduled_departure,source_snapshot
15129,M1,2026-01-05,7118,212.0,295.983333,33.575472,manhattan_20251230
15130,M10,2026-01-05,2691,155.0,133.983333,17.361290,manhattan_20251230
15131,M101,2026-01-05,12541,329.0,503.366667,38.118541,manhattan_20251230
15132,M102,2026-01-05,7468,222.0,274.633333,33.639640,manhattan_20251230
15133,M103,2026-01-05,6343,175.0,241.216667,36.245714,manhattan_20251230
15134,M104,2026-01-05,4676,192.0,176.683333,24.354167,manhattan_20251230
15135,M106,2026-01-05,1332,86.0,36.150000,15.488372,manhattan_20251230
15136,M11,2026-01-05,7942,185.0,230.000000,42.929730,manhattan_20251230
15137,M116,2026-01-05,4499,243.0,109.316667,18.514403,manhattan_20251230
15138,M12,2026-01-05,1026,64.0,32.216667,16.031250,manhattan_20251230


## 7. Structural route coverage and overlap

Use the explicitly versioned September archive, restricted to trip-referenced bus routes.
Served stops are unique IDs; pairwise Jaccard = intersection/union, while smaller-route overlap
= intersection/minimum set size. Preserve exclusive stops as a coverage safeguard.

Shared IDs indicate common stops, not shared road length, interchangeable services or a valid
merge. Nearby opposite-side stops can have different IDs; stop-level demand and walking access
are unavailable. This is structural evidence for review only.

In [8]:
selected_id = 'manhattan_20260903'
selected_trips = trip_features.loc[trip_features.snapshot_id.eq(selected_id)]
route_stops = selected_geometry['stop_times'].merge(selected_trips[['trip_id','bus_route']],on='trip_id',validate='many_to_one')
route_stops = route_stops[['bus_route','stop_id']].drop_duplicates()
stop_sets = route_stops.groupby('bus_route').stop_id.agg(set)
stop_routes = route_stops.groupby('stop_id').bus_route.nunique().rename('routes_serving_stop')
route_structure = selected_trips.groupby('bus_route').agg(
    trip_templates=('trip_id','size'), stop_patterns=('pattern_id','nunique'),
    mean_template_duration_minutes=('scheduled_duration_minutes','mean'),
    mean_template_shape_length_km=('analysis_shape_length_km','mean'),
    mean_template_stop_visits=('stop_visits','mean')).reset_index()
route_structure['served_stop_ids'] = route_structure.bus_route.map(stop_sets.map(len))
exclusive = route_stops.loc[route_stops.stop_id.map(stop_routes).eq(1)].groupby('bus_route').size()
route_structure['exclusive_stop_ids'] = route_structure.bus_route.map(exclusive).fillna(0).astype(int)
route_structure['snapshot_id'] = selected_id
from itertools import combinations
overlap_rows = []
for (route_a,stops_a),(route_b,stops_b) in combinations(stop_sets.items(),2):
    shared = len(stops_a & stops_b)
    overlap_rows.append({'route_a':route_a,'route_b':route_b,'shared_stop_ids':shared,
        'jaccard':shared/len(stops_a|stops_b),'smaller_route_overlap':shared/min(len(stops_a),len(stops_b)),
        'a_only_stop_ids':len(stops_a-stops_b),'b_only_stop_ids':len(stops_b-stops_a),'snapshot_id':selected_id})
route_overlap = pd.DataFrame(overlap_rows)
map_stops = selected_geometry['stops'].merge(stop_routes,on='stop_id',how='inner',validate='one_to_one')
for col in ['stop_lat','stop_lon']: map_stops[col] = pd.to_numeric(map_stops[col],errors='raise')
display(route_structure.head(8))
display(route_overlap.sort_values(['jaccard','shared_stop_ids'],ascending=False).head(10))
print('All route pairs assessed, including zero overlap:',len(route_overlap))

,bus_route,trip_templates,stop_patterns,mean_template_duration_minutes,mean_template_shape_length_km,mean_template_stop_visits,served_stop_ids,exclusive_stop_ids,snapshot_id
0,M1,932,6,77.049356,13.287595,63.862661,134,36,manhattan_20260903
1,M10,773,3,49.394567,8.343840,43.902975,88,78,manhattan_20260903
2,M101,1396,14,87.960602,15.508941,52.985673,151,40,manhattan_20260903
3,M102,1029,6,69.525753,11.244043,51.305151,113,2,manhattan_20260903
4,M103,814,4,77.925061,12.599032,52.260442,105,20,manhattan_20260903
5,M104,1146,2,52.304538,8.295721,38.475567,77,42,manhattan_20260903
6,M106,338,2,24.127219,3.952609,16.500000,33,12,manhattan_20260903
7,M11,791,9,71.243995,11.945386,57.077118,121,71,manhattan_20260903


,route_a,route_b,shared_stop_ids,jaccard,smaller_route_overlap,a_only_stop_ids,b_only_stop_ids,snapshot_id
649,M34+,M34A+,22,0.536585,0.758621,12,7,manhattan_20260903
117,M102,M103,76,0.535211,0.723810,37,29,manhattan_20260903
80,M101,M103,81,0.462857,0.771429,70,24,manhattan_20260903
14,M1,M2,83,0.425641,0.619403,51,61,manhattan_20260903
79,M101,M102,75,0.396825,0.663717,76,38,manhattan_20260903
499,M2,M3,79,0.348018,0.548611,65,83,manhattan_20260903
19,M1,M3,70,0.309735,0.522388,64,92,manhattan_20260903
638,M31,M57,23,0.302632,0.547619,34,19,manhattan_20260903
385,M14A+,M14D+,19,0.279412,0.452381,23,26,manhattan_20260903
257,M106,M96,12,0.266667,0.500000,21,12,manhattan_20260903


All route pairs assessed, including zero overlap: 820


## 8. Route/day-type profiles use ratios of totals

Aggregate only eligible pairs and keep date counts. Never average daily boarding/trip ratios.
No hourly demand is invented from daily totals. All descriptive profiles use the stated observed
period; future model evaluation must rebuild them inside training windows to avoid leakage.

In [9]:
eligible = matched_daily.loc[matched_daily.comparison_eligible & matched_daily.scheduled_departures.gt(0)
                            & matched_daily.scheduled_service_hours.gt(0)].copy()
profiles = eligible.groupby(['bus_route','day_type'],as_index=False).agg(
    observed_dates=('payment_date','nunique'),first_date=('payment_date','min'),last_date=('payment_date','max'),
    total_boardings=('analysis_boardings','sum'),total_scheduled_departures=('scheduled_departures','sum'),
    total_scheduled_service_hours=('scheduled_service_hours','sum'),mean_daily_boardings=('analysis_boardings','mean'),
    std_daily_boardings=('analysis_boardings','std'))
profiles['boardings_per_scheduled_departure'] = profiles.total_boardings/profiles.total_scheduled_departures
profiles['boardings_per_scheduled_service_hour'] = profiles.total_boardings/profiles.total_scheduled_service_hours
profiles['daily_boardings_cv'] = profiles.std_daily_boardings/profiles.mean_daily_boardings.replace(0,np.nan)
profiles['enough_dates_for_review'] = profiles.observed_dates.ge(20)
display(profiles.sort_values('boardings_per_scheduled_service_hour',ascending=False).head(10))
print('Undefined one-departure headways:',int(patterns_daily.mean_headway_minutes.isna().sum()))
print('Pattern groups with one departure:',int(patterns_daily.scheduled_departures.eq(1).sum()))

,bus_route,day_type,observed_dates,first_date,last_date,total_boardings,total_scheduled_departures,total_scheduled_service_hours,mean_daily_boardings,std_daily_boardings,boardings_per_scheduled_departure,boardings_per_scheduled_service_hour,daily_boardings_cv,enough_dates_for_review
113,M86+,Weekday,181,2026-01-05,2026-09-30,2417059.0,72638.0,26454.633333,13353.917127,1955.165812,33.275407,91.366188,0.146411,True
68,M34+,Weekday,181,2026-01-05,2026-09-30,866228.0,30855.0,11150.566667,4785.790055,835.078273,28.074153,77.684662,0.174491,True
111,M86+,Saturday,33,2026-01-17,2026-09-26,264712.0,10270.0,3737.700000,8021.575758,865.362534,25.775268,70.822163,0.107879,True
59,M23+,Weekday,181,2026-01-05,2026-09-30,1386101.0,49832.0,19671.133333,7658.016575,1192.183233,27.815480,70.463708,0.155678,True
71,M34A+,Weekday,181,2026-01-05,2026-09-30,841329.0,29873.0,12213.000000,4648.226519,733.925086,28.163526,68.887988,0.157894,True
67,M34+,Sunday,32,2026-01-18,2026-09-27,66799.0,2944.0,982.933333,2087.468750,601.255406,22.689878,67.958831,0.288031,True
112,M86+,Sunday,32,2026-01-18,2026-09-27,202415.0,8576.0,2990.400000,6325.468750,1109.676529,23.602495,67.688269,0.175430,True
119,M96,Weekday,181,2026-01-05,2026-09-30,1734290.0,74094.0,25803.250000,9581.712707,1557.557056,23.406619,67.212076,0.162555,True
80,M42,Weekday,181,2026-01-05,2026-09-30,1282234.0,51880.0,19531.300000,7084.165746,1103.868840,24.715382,65.650213,0.155822,True
66,M34+,Saturday,33,2026-01-17,2026-09-26,84999.0,3626.0,1303.583333,2575.727273,486.201750,23.441533,65.204117,0.188763,True


Undefined one-departure headways: 1027
Pattern groups with one departure: 1027


## 9. Save versioned products and expose limitations

The conservative seven-route audit uses a service-date screen. The comparisons in this
notebook use an exact route-ID crosswalk and clock-date service integration. Their counts
describe different eligibility rules; both sets of results retain their source provenance.

`comparison_eligible` is an analytical screen, not a correctness guarantee. Current shape profiles
are not automatically historical predictor features. Demand lags require a publication-delay policy
before operational forecasting. Zero occupancy, missing delays and unavailable capacity are not features.

In [10]:
products = {'daily_demand_features':daily_features,'trip_template_features':trip_features,
    'shape_features':shape_features,'calendar_expansion':calendar_audit,
    'pattern_service_date_features':patterns_daily,'schedule_clock_daily':schedule_daily,
    'schedule_clock_hourly':grid,'matched_daily_features':matched_daily,'matched_hourly_features':matched_hourly,
    'route_day_type_profiles':profiles,'route_structure':route_structure,'route_overlap':route_overlap,
    'map_stops':map_stops,'map_shapes':selected_geometry['shapes'],
    'map_route_shapes':selected_trips[['bus_route','shape_id']].drop_duplicates()}
schema_rows=[]
for name,frame in products.items():
    frame.to_csv(OUT/f'{name}.csv',index=False)
    for column in frame:
        schema_rows.append({'table':name,'column':column,'dtype':str(frame[column].dtype),
                            'missing_count':int(frame[column].isna().sum())})
pd.DataFrame(schema_rows).to_csv(REPORT/'feature_schema.csv',index=False)
feature_manifest={'inputs':{'cleaning_manifest_sha256':hashlib.sha256((OUT/'cleaning_manifest.json').read_bytes()).hexdigest()},
    'parameters':{'first_schedule_date':str(first_date.date()),'last_schedule_date':str(last_date.date()),
                  'mapping':'exact source route_id','snapshot_selection':'latest whole capture; no fallback',
                  'timezone':'America/New_York','dst_policy':'withhold transition and neighbour clock dates',
                  'profile_minimum_review_dates':20,'shape_segment_review_km':5},
    'products':{name:{'rows':len(frame),'columns':list(frame.columns),
        'sha256':hashlib.sha256((OUT/f'{name}.csv').read_bytes()).hexdigest()} for name,frame in products.items()}}
(OUT/'feature_manifest.json').write_text(json.dumps(feature_manifest,indent=2),encoding='utf-8')
display(pd.DataFrame([{'product':name,'rows':len(frame),'columns':frame.shape[1]} for name,frame in products.items()]))
print('Saved feature tables and manifest. No model, fitted preprocessor or crowding label created.')

,product,rows,columns
0,daily_demand_features,275896,23
1,trip_template_features,174174,27
2,shape_features,784,6
3,calendar_expansion,275,9
4,pattern_service_date_features,36181,22
5,schedule_clock_daily,11193,10
6,schedule_clock_hourly,268632,15
7,matched_daily_features,26158,34
8,matched_hourly_features,90528,26
9,route_day_type_profiles,121,14


Saved feature tables and manifest. No model, fitted preprocessor or crowding label created.
